# SplitSnap 3: results and figures

Reads the measured results stored in `results/` and redraws the main tables and figures. Nothing is computed by a model here; every number comes from a file produced by the scripts named in `results/README.md`. Numbers on generated bills are not real-bill results.

[Open in Colab](https://colab.research.google.com/github/atrijopal/naala_ml_tharki_ap/blob/main/notebooks/03_results_and_figures.ipynb)

In [ ]:
import ast, json, os, subprocess, sys, pathlib

if "google.colab" in sys.modules or os.path.exists("/kaggle"):
    if not os.path.exists("proj"):
        subprocess.run("git clone -q https://github.com/atrijopal/naala_ml_tharki_ap.git proj", shell=True, check=True)
    os.chdir("proj")
root = pathlib.Path.cwd()
while not (root / "results").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
import matplotlib.pyplot as plt
load = lambda p: json.load(open(p))
print("results folder:", sorted(os.listdir("results"))[:6], "...")

## Training: loss and validation scores

In [ ]:
rows = {}
for f in ("results/training_logs/stage1_epochs1-8.log", "results/training_logs/stage2_epochs9-16.log"):
    for line in open(f, errors="ignore"):
        if line.startswith("{'epoch'"):
            d = ast.literal_eval(line.strip())
            if d.get("epoch_sec", 0) > 100:                     # the real run, not the short overfit gate
                rows[d["epoch"]] = d
ep = sorted(rows)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].semilogy(ep, [rows[e]["train_loss"] for e in ep], "o-", color="#a23b2a"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss"); ax[0].grid(alpha=.3)
val = [e for e in ep if "val" in rows[e]]
ax[1].plot(val, [rows[e]["val"]["cord"]["ted_acc"] for e in val], "s-", color="black", label="CORD edit-distance score")
ax[1].plot(val, [rows[e]["val"]["sroie"]["score"] for e in val], "o--", color="#a23b2a", label="SROIE field score")
ax[1].plot(val, [rows[e]["val"]["synth"]["ted_acc"] for e in val], "^:", color="#5e6b2d", label="generated bills")
ax[1].set_xlabel("epoch"); ax[1].set_ylim(0.8, 1.0); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.show()

## Held-out test results and before/after fine-tuning

In [ ]:
m = load("results/heldout_test_predictions.json")["metrics"]
for src, v in m.items():
    print(src, {k: (round(x, 3) if isinstance(x, float) else x) for k, x in v.items() if not isinstance(x, (dict, list))})
ba = load("results/before_after.json")
print(f"\nBefore and after fine-tuning on {ba['n']} unseen generated bills")
for k in ("ted_acc", "field_f1", "item_f1", "charge_f1", "grand_total_acc"):
    print(f"  {k:16s} {ba['before'][k]:.3f} -> {ba['after'][k]:.3f}")

## Robustness to poor photos (generated bills, controlled degradation)

In [ ]:
C = load("results/degradation.json")["conditions"]
lab = lambda c: "clean" if c["kind"] == "clean" else f"{c['kind']} {c['level']:g}"
x = range(len(C))
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(x, [c["field_f1"] for c in C], color="#5e6b2d", label="field F1")
ax.plot(x, [c["errors_caught"] or 0 for c in C], "o", color="#a23b2a", label="wrong fields the flags catch")
ax.plot(x, [c["retake_share"] for c in C], "s", color="#b07a1e", label="bills the app asks to retake")
ax.set_xticks(list(x)); ax.set_xticklabels([lab(c) for c in C], rotation=30, ha="right"); ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=.3); ax.legend(ncol=3, loc="lower left")
plt.tight_layout(); plt.show()
for r in load("results/deskew_check.json"):
    print(f"tilt {r['tilt_deg']:2d} deg  {r['mode']:9s} field F1 {r['field_f1']:.3f}  item F1 {r['item_f1']:.3f}")

## Confidence: what highlighting more fields buys

In [ ]:
K = load("results/calibration.json")
t = K["tradeoff"]
plt.figure(figsize=(5.5, 3.4))
plt.plot([100 * r["flagged_share"] for r in t], [100 * r["errors_caught"] for r in t], "o-", color="#a23b2a", label="real errors caught")
plt.plot([100 * r["flagged_share"] for r in t], [100 * r["precision_of_unflagged"] for r in t], "s--", color="#5e6b2d", label="unhighlighted fields correct")
plt.axvline(8, color="black", ls=":"); plt.xlabel("share of fields highlighted, %"); plt.ylabel("%"); plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()
print({k: (v["auroc"], v["ece_raw"]) for k, v in K["aggregators"].items()})

## Mistakes on clean bills, and the real-scan check

In [ ]:
E = load("results/error_analysis.json")
print("bills with a mistake:", E["bills_with_any_error"], "of", E["bills"], "| wrong fields:", E["errors_total"])
print({k: v for k, v in E["error_counts"].items() if v})
print("rate suggestion:", load("results/repair_check.json"))
S = load("results/sroie_scan_check.json")["summary"]
print("\nreal SROIE scans, main task returns any item (of", S["scans"], "):", S["main_task_nonempty"])
print("auxiliary SROIE task exact matches:", S["aux_task_fields_exact"])

## Speed and capacity

In [ ]:
for r in load("results/load_test.json")["levels"]:
    print(f"{r['clients']} clients: {r['bills_per_s']:.2f} bills/s, p50 {r['latency_p50_s']:.2f} s, p95 {r['latency_p95_s']:.2f} s, validate {r['validate_probe_median_ms']:.0f} ms")